In [ ]:
#Spark connection
import os
import socket
from pyspark.sql import SparkSession
 
APACHE_MASTER_IP = socket.gethostbyname("apache-spark-master-0.apache-spark-headless.apache-spark.svc.cluster.local")
APACHE_MASTER_URL = f"spark://{APACHE_MASTER_IP}:7077"
POD_IP = os.environ["MY_POD_IP"]
SPARK_APP_NAME = f"spark-{os.environ['HOSTNAME']}"
JARS = "/nfs/env/lib/python3.8/site-packages/pyspark/jars/clickhouse-native-jdbc-shaded-2.6.5.jar"
MEM = "512m"
CORES = 1
 
spark = SparkSession.\
        builder.\
        appName(SPARK_APP_NAME).\
        master(APACHE_MASTER_URL).\
        config("spark.executor.memory", MEM).\
        config("spark.jars", JARS).\
        config("spark.executor.cores", CORES).\
        config("spark.driver.host", POD_IP).\
        getOrCreate()

In [ ]:
import pandas as pd

pd_orders = pd.read_csv('orders.csv')  

In [ ]:
df_orders = spark.createDataFrame(
    (pd_orders), 
    schema = 'order_id INT, order_date STRING, order_customer_id INT, order_status STRING'
)

df_orders.printSchema()

In [ ]:
from pyspark.sql.functions import date_format

df_orders.show()

In [ ]:
#Добавляем новую колонку order_month с использованием select и метода data_format
#https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.date_format.html#pyspark.sql.functions.date_format

#Список всех функций
#https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html

df_orders.select('*', date_format('order_date', 'yyyyMM').alias('order_month')).show()

In [ ]:
#Повторяем, но с использованием withColumn
df_orders.withColumn('order_month', date_format('order_date', 'yyyyMM')).show()

In [ ]:
#Пример date_format вместе с filter. Фактически мы задаем условие where
 
df_orders. \
    filter(date_format('order_date', 'yyyyMM') == 201401). \
    show()

In [ ]:
#Пример с groupBy
 
df_orders. \
    groupBy(date_format('order_date', 'yyyyMM').alias('order_month')). \
    count(). \
    show()

In [ ]:
spark.stop()